In [ ]:
import httpx
import pandas as pd

from aare_train.constants import LOC_BERN

# Bafu Hydro Data scraping

Figuring out how to get the bafu flow forecasts from their site.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (16, 9)

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
client = httpx.AsyncClient()

In [ ]:
loc = LOC_BERN
url = f"https://www.hydrodaten.admin.ch/plots/q_forecast/{loc}_q_forecast_de.json"
r = await client.get(url)
body = r.json()

In [ ]:
data = body["plot"]["data"]
data

In [ ]:
def extract(data: dict[str, str | float]):
    data = data["x"], data["y"]
    data = pd.DataFrame(dict(time=pd.to_datetime(data[0]), flow_bern=data[1]))

    return data

In [ ]:
median = extract(data[3])
median

In [ ]:
d0 = extract(data[0])
d0

In [ ]:
d1 = extract(data[1])
d1

In [ ]:
(d0 <= d1).value_counts()

In [ ]:
(d1 <= d0).value_counts()

In [ ]:
# it seems d1 is min and d0 is max because d1 is always <= d0, but no the other way around

In [ ]:
def trav(obj, *path):
    cur = obj
    for seg in path:
        cur = cur[seg]

    return cur


def _get_x_y(trace: dict[str, str | float]) -> tuple[list[str], list[float]]:
    x = trav(trace, "x")
    y = trav(trace, "y")
    assert isinstance(x, list), "x is not a list"
    assert isinstance(y, list), "y is not a list"
    return x, y


trace = data[2]

base_col_name = "flow_bern"
q25_suffix = "_q25"
q75_suffix = "_q75"
x, y = _get_x_y(trace)
half, rest = divmod(len(x), 2)

# for some weird reason, it seems that the last datapoint is (always?) duplicated
x1, x2 = x[:half], x[half : len(x) - rest][::-1]
y1, y2 = y[:half], y[half : len(y) - rest][::-1]

times = pd.to_datetime(x1)
if not (times == pd.to_datetime(x2)).all():
    raise ValueError("Times in first and second half aren't aligned")

pd.DataFrame({"time": times, base_col_name + q25_suffix: y1, base_col_name + q75_suffix: y2})

In [ ]:
pd.DataFrame(dict(x1=pd.to_datetime(x1), x2=pd.to_datetime(x2[::-1])))

In [ ]:
len(x)

In [ ]:
x

In [ ]:
half = 118
x1 = x[:half]
print(len(x1))
print(len(set(x1)))
x1

In [ ]:
x2 = x[half:][::-1]
print(len(x2))
print(len(set(x1)))
x2

In [ ]:
set(x1).symmetric_difference(set(x2))